# scatE classifier — diagnostics

Plotting and diagnostic tools for the `scatE` MLP. Loads a saved checkpoint from
[train_scatE.py](train_scatE.py), runs it over the held-out test file, and draws:

1. the truth kinematics pair plot — `p` / `theta` / `phi` split by `scatE`,
2. the confusion matrix,
3. the ROC curve,
4. the efficiency profiles and 2-D closure maps in `p` / `theta` / `phi`.

The confusion matrix uses **Bernoulli sampling**, not a fixed threshold: the model output is a
reconstruction *efficiency*, so a prediction is drawn as `scatE ~ Bernoulli(p)` per event — the same
way a fast simulation would consume it. Everything reusable (loading, `MLP`, `predict_logits`) is
imported from `train_scatE`; this notebook only defines plots.

In [ ]:
import os
from pathlib import Path

import awkward as ak
import numpy as np
import torch
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

import train_scatE as t

import matplotlib.pyplot as plt

# train_scatE forces the Agg backend on import, which would make every figure below blank.
# This magic must come *after* that import.
%matplotlib inline

# Paths are relative to the repository root. Set PARNASSUS_DATA to the directory that
# holds v8/v8_test.root, and PARNASSUS_FIG_DIR to change where figures are written.
REPO_ROOT = Path("../../../../..")  # scatE_classifier/ -> repository root
DATA_DIR = Path(os.environ.get("PARNASSUS_DATA", REPO_ROOT / ".." / "parnassus_data"))
FIG_DIR = Path(os.environ.get("PARNASSUS_FIG_DIR", REPO_ROOT / "plots" / "paper_figs"))
FIG_DIR.mkdir(parents=True, exist_ok=True)
PRETRAINED = REPO_ROOT / "src/parnassus_core/experiments/clas12_RGA/pre-trained-models"

CFG = {
    "root_file": str(DATA_DIR / "v8" / "v8_test.root"),
    "tree": "evt_tree",
    "n_events": 2_000_000,  # the test file holds 29_910_000; 2M is plenty for diagnostics
    # the paper checkpoint; use trained_models/classifier/best_model.ckpt for a new run
    "ckpt": str(PRETRAINED / "pre-filter_model.ckpt"),
    "eval_batch_size": 65536,
    "seed": 0,  # seeds the Bernoulli draw
    "n_scatter": 5000,  # points per class in the pair-plot scatter panels
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")

plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Helvetica", "Arial", "Liberation Sans"]
plt.rcParams["font.size"] = 14
plt.rcParams["axes.labelsize"] = 10
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["xtick.labelsize"] = 14
plt.rcParams["ytick.labelsize"] = 14

In [ ]:
def load_model(ckpt_path, device):
    """Rebuild the trained MLP from a checkpoint. Returns (model, ckpt).

    weights_only=False is required: the checkpoint carries config/feature_mean/feature_std
    alongside the state dict, and torch 2.6+ defaults to weights_only=True.
    """
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model = t.MLP(
        n_inputs=len(t.INPUT_FIELDS),
        hidden=ckpt["config"]["hidden"],
        dropout=ckpt["config"]["dropout"],
    ).to(device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    return model, ckpt


model, ckpt = load_model(CFG["ckpt"], device)
print(f"{CFG['ckpt']}")
print(f"best epoch {ckpt['epoch']}   val loss {ckpt['val_loss']:.4f}")
print(f"hidden {tuple(ckpt['config']['hidden'])}   dropout {ckpt['config']['dropout']}")
print(f"feature mean {ckpt['feature_mean']}   std {ckpt['feature_std']}")
model

In [ ]:
# Features and labels for the whole file - no train/val/test split here, this file *is* the
# held-out set, and the checkpoint carries its own standardization so it stands alone.
pflow_labels, truth_electrons = t.load_events(CFG)

X = np.stack([ak.to_numpy(truth_electrons[f]) for f in t.INPUT_FIELDS], axis=1).astype(
    np.float32
)
y = pflow_labels["scatE"].astype(np.float32)

# p = pt/sin(theta) can blow up at small theta - drop those rows from features AND labels
# together (load_events does not do this; prepare_tensors is where it lives in the script).
good = np.isfinite(X).all(axis=1)
X, y = X[good], y[good]
print(f"dropped {int(np.sum(~good))} events with non-finite features")

X_t = torch.from_numpy((X - ckpt["feature_mean"]) / ckpt["feature_std"]).to(device)
probs = (
    torch.sigmoid(t.predict_logits(model, X_t, CFG["eval_batch_size"])).cpu().numpy()
)
y_true = y.astype(int)

print(f"events: {len(y_true)}   class balance: scatE==1 -> {y_true.mean() * 100:.2f}%")
print(f"mean predicted probability: {probs.mean() * 100:.2f}%")

AXIS_LABELS = {"p": "p [GeV]", "theta": "theta [rad]", "phi": "phi [rad]"}

In [ ]:
def plot_kinematics(X, y, field_names, n_scatter=5000, bins=50):
    """Lower-triangle pair plot of the truth kinematics, split by scatE.

    Diagonal: overlaid 1D histograms over all events.
    Lower triangle: 2D scatter of the first n_scatter events of each class.
    """
    n = len(field_names)
    pos, neg = X[y == 1], X[y == 0]
    # majority class first so the ~19% reconstructed population is drawn on top of it
    populations = [(neg, "$e^-$ NOT Reconstructed"), (pos, "$e^-$ Reconstructed")]

    fig, axs = plt.subplots(n, n, figsize=(10, 10))
    for i in range(n):
        for j in range(n):
            ax = axs[i, j]
            if i < j:
                ax.remove()
                continue
            if i == j:
                for data, label in populations:
                    ax.hist(data[:, i], bins=bins, alpha=0.5, label=label, density=True)
                ax.set_xlabel(AXIS_LABELS[field_names[i]])
                ax.set_ylabel("Normalized Counts")
                continue
            for data, label in populations:
                sub = data[:n_scatter]
                ax.plot(sub[:, i], sub[:, j], "o", alpha=0.4, ms=1.5, label=label)
            ax.set_xlabel(AXIS_LABELS[field_names[i]])
            ax.set_ylabel(AXIS_LABELS[field_names[j]])

    fig.suptitle("Comparison of Generated Scattered $e^-$ Kinematics ", fontsize=22)
    fig.tight_layout()

    # the vacated upper triangle is the natural home for the legend
    handles, labels = axs[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="center", bbox_to_anchor=(0.77, 0.9), fontsize=18)
    return fig


plot_kinematics(X, y, t.INPUT_FIELDS, n_scatter=CFG["n_scatter"])
plt.show()

In [ ]:
def bernoulli_predict(probs, seed=0):
    """Sample scatE ~ Bernoulli(p) per event - the operating point a fast sim would use."""
    rng = np.random.default_rng(seed)
    return (rng.random(len(probs)) < probs).astype(int)


def plot_confusion_matrix(cm, title_suffix=""):
    """Counts and truth-row-normalized confusion matrix side by side."""
    fig, axs = plt.subplots(1, 2, figsize=(11, 4.5))
    ConfusionMatrixDisplay(cm, display_labels=["scatE=0", "scatE=1"]).plot(
        ax=axs[0], colorbar=False
    )
    axs[0].set_title(f"Counts{title_suffix}")
    ConfusionMatrixDisplay(
        cm / cm.sum(axis=1, keepdims=True), display_labels=["scatE=0", "scatE=1"]
    ).plot(ax=axs[1], colorbar=False, values_format=".3f")
    axs[1].set_title(f"Normalized by truth row{title_suffix}")
    fig.tight_layout()
    return fig


y_pred = bernoulli_predict(probs, seed=CFG["seed"])
cm = confusion_matrix(y_true, y_pred)

acc = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
majority = int(round(y_true.mean()))
baseline_acc = accuracy_score(y_true, np.full_like(y_true, majority))

print(f"test events: {len(y_true)}   Bernoulli sampling (seed {CFG['seed']})")
# Lower than the thresholded numbers (acc 0.9475 / F1 0.8702 at threshold 0.5) by construction:
# a randomized classifier gives up per-event accuracy to get the output *distribution* right.
print(f"accuracy: {acc:.4f}")
print(f"precision: {precision:.4f}")
print(f"recall: {recall:.4f}")
print(f"f1 score: {f1:.4f}")
print(f"confusion matrix (rows = truth, cols = predicted):\n{cm}")
print(f"majority-class baseline accuracy ({majority}): {baseline_acc:.4f}")
# The reason for sampling instead of thresholding: these two should agree closely.
print(
    f"\npositive rate  sampled: {y_pred.mean() * 100:.2f}%   truth: {y_true.mean() * 100:.2f}%"
)

plot_confusion_matrix(cm, title_suffix=" (Bernoulli)")
plt.show()

In [ ]:
def plot_roc_curve(y_true, probs, cm, auc):
    """ROC curve with the realized Bernoulli operating point marked.

    The marker sits strictly inside the curve, and that is not a bug: a randomized classifier is
    always dominated in ROC space by the best deterministic threshold.
    """
    fpr, tpr, _ = roc_curve(y_true, probs)
    tn, fp, fn, tp = cm.ravel()
    op_fpr, op_tpr = fp / (fp + tn), tp / (tp + fn)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(fpr, tpr, label=f"MLP (AUC = {auc:.4f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="random guess (AUC = 0.5)")
    ax.plot(
        op_fpr,
        op_tpr,
        "o",
        ms=7,
        color="tab:red",
        label=f"Bernoulli draw (TPR {op_tpr:.3f}, FPR {op_fpr:.3f})",
    )
    ax.set_xlabel("False positive rate  (scatE=0 called 1)")
    ax.set_ylabel("True positive rate  (scatE=1 found)")
    ax.set_title("ROC curve - scatE classification (test set)")
    ax.legend(loc="lower right")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    return fig


auc = roc_auc_score(y_true, probs)
print(f"AUC: {auc:.4f}")

plot_roc_curve(y_true, probs, cm, auc)
plt.show()

## Efficiency profiles

The model output *is* a reconstruction efficiency, so the sharpest check is whether it reproduces the
measured `scatE` rate as a function of kinematics. Below: the measured rate (binomial errors) against
the model's mean predicted probability, binned in each of `p`, `theta`, `phi`, then the same thing as
a residual, then the 2-D closure maps.

The revealing feature is **`phi`**: CLAS12 should show six-sector structure (six torus coils
$\rightarrow$ six dips). `phi` is fed to the model as a raw angle, so if the curve smooths those dips
over, that is the argument for a `sin(phi)`/`cos(phi)` encoding. In `theta` the interesting places are
the edges, where the forward-detector acceptance turns on and off.

In [ ]:
# Binning helpers, shared by the 1-D profile, the residual and the 2-D maps below.
BINS_1D = {"p": 60, "theta": 60, "phi": 72}  # 72 phi bins = 12 per CLAS12 sector


def binned(x, n_bins):
    """Bin index, bin centers and per-bin counts for a feature column."""
    edges = np.histogram_bin_edges(x, bins=n_bins)
    idx = np.clip(np.digitize(x, edges[1:-1]), 0, n_bins - 1)
    return (
        idx,
        0.5 * (edges[:-1] + edges[1:]),
        np.bincount(idx, minlength=n_bins).astype(float),
    )


def bin_mean(idx, values, n):
    """Mean of `values` per bin; empty bins come back as 0 and are masked by the caller."""
    return np.bincount(idx, weights=values, minlength=len(n)) / np.maximum(n, 1)


def plot_efficiency_profiles(X, y_true, probs, field_names, bins_per_field):
    """Measured scatE rate vs the model's mean predicted probability, one panel per feature."""
    fig, axs = plt.subplots(len(field_names), 1, figsize=(10, 11))
    for ax, (i, field) in zip(axs, enumerate(field_names)):
        idx, centers, n = binned(X[:, i], bins_per_field[field])
        nz = n > 0

        obs = bin_mean(idx, y_true, n)
        err = np.sqrt(np.maximum(obs * (1 - obs), 0) / np.maximum(n, 1))  # binomial

        ax.errorbar(
            centers[nz],
            obs[nz],
            yerr=err[nz],
            fmt="ko",
            ms=4,
            lw=1,
            label="clasdis $e^-$ reconstruction efficiency",
            zorder=3,
        )
        ax.plot(
            centers[nz],
            bin_mean(idx, probs, n)[nz],
            "-",
            ms=3,
            color="tab:orange",
            lw=1.6,
            label="Filter (MLP) mean prediction",
        )

        ax.set_xlabel(AXIS_LABELS[field])
        ax.set_ylabel("P($e^-$ Reconstruction)")
        ax.set_title(f"Efficiency vs {field}")

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        if field == "theta":
            ax.set_xlim(0.11, 0.7)
        # ax.grid(alpha=0.3)
    axs[0].legend()
    fig.tight_layout()
    return fig


plot_efficiency_profiles(X, y_true, probs, t.INPUT_FIELDS, BINS_1D)
plt.savefig(FIG_DIR / "Filter_Efficiency.pdf")
plt.show()

In [ ]:
def plot_closure_residuals(X, y_true, probs, field_names, bins_per_field):
    """Model - measured, per feature: the same information as the overlay above, but readable.

    The grey band is the +/-1 sigma statistical uncertainty on the measured rate, so a curve inside
    the band is as close as this many events can resolve.
    """
    fig, axs = plt.subplots(len(field_names), 1, figsize=(10, 10))
    for ax, (i, field) in zip(axs, enumerate(field_names)):
        idx, centers, n = binned(X[:, i], bins_per_field[field])
        nz = n > 0

        obs = bin_mean(idx, y_true, n)
        err = np.sqrt(np.maximum(obs * (1 - obs), 0) / np.maximum(n, 1))
        resid = bin_mean(idx, probs, n) - obs
        rms = np.sqrt(np.average(resid[nz] ** 2, weights=n[nz]))

        ax.plot(
            centers[nz],
            resid[nz],
            "-o",
            ms=3,
            color="tab:blue",
            label=f"MLP (weighted RMS {rms:.4f})",
        )
        ax.fill_between(
            centers[nz],
            -err[nz],
            err[nz],
            color="grey",
            alpha=0.3,
            label="+/- 1 sigma statistical",
        )
        ax.axhline(0, color="k", lw=1)
        ax.set_xlabel(AXIS_LABELS[field])
        ax.set_ylabel("mean predicted - measured")
        ax.set_title(f"Closure residual vs {field}")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    fig.tight_layout()
    return fig


plot_closure_residuals(X, y_true, probs, t.INPUT_FIELDS, BINS_1D)
plt.show()

In [ ]:
# 2-D efficiency maps, one row per pair of features.
#
# The 1-D panels above are *marginals*: a phi bin holds every value of p and theta, so an
# over-prediction at low theta can cancel an under-prediction at high theta and the model looks
# perfect. These maps do not let that happen, which is why their RMS values are the larger ones.
from itertools import combinations

BINS_2D = {"p": 40, "theta": 40, "phi": 36}  # 36 phi bins = 6 per CLAS12 sector
MIN_COUNT = 100  # sparser bins are masked: efficiency too noisy to read


def efficiency_map(X, y_true, probs, ix, iy):
    """Bin edges, counts, measured efficiency and mean predicted probability, in 2-D."""
    fx, fy = t.INPUT_FIELDS[ix], t.INPUT_FIELDS[iy]
    xe = np.histogram_bin_edges(X[:, ix], bins=BINS_2D[fx])
    ye = np.histogram_bin_edges(X[:, iy], bins=BINS_2D[fy])
    cols = (X[:, ix], X[:, iy])
    n = np.histogram2d(*cols, bins=(xe, ye))[0]
    sparse = n < MIN_COUNT

    def rate(weights):
        s = np.histogram2d(*cols, bins=(xe, ye), weights=weights)[0]
        return np.where(sparse, np.nan, s / np.maximum(n, 1))

    return xe, ye, n, rate(y_true), rate(probs)


def draw_map(ax, xe, ye, arr, ix, iy, cmap, **kw):
    cmap = plt.get_cmap(cmap).copy()
    cmap.set_bad("0.85")  # grey = too few events to measure
    im = ax.pcolormesh(xe, ye, arr.T, cmap=cmap, shading="flat", **kw)
    ax.set_xlabel(AXIS_LABELS[t.INPUT_FIELDS[ix]])
    ax.set_ylabel(AXIS_LABELS[t.INPUT_FIELDS[iy]])
    return im


pairs = list(
    combinations(range(len(t.INPUT_FIELDS)), 2)
)  # (p,theta), (p,phi), (theta,phi)
maps = {pair: efficiency_map(X, y_true, probs, *pair) for pair in pairs}

# One residual colour scale for every panel, clipped at the 99th percentile so that a handful of
# extreme bins cannot flatten the rest.
allr = np.concatenate(
    [
        (pred - obs)[np.isfinite(pred - obs)].ravel()
        for _, _, _, obs, pred in maps.values()
    ]
)
lim = float(np.percentile(np.abs(allr), 99))

fig, axs = plt.subplots(len(pairs), 3, figsize=(16, 4.3 * len(pairs)))
for row, pair in enumerate(pairs):
    xe, ye, n, obs, pred = maps[pair]
    vmax = float(np.nanmax(obs))
    for col, (title, arr) in enumerate([("measured", obs), ("MLP", pred)]):
        im = draw_map(axs[row, col], xe, ye, arr, *pair, "viridis", vmin=0, vmax=vmax)
        axs[row, col].set_title(f"{title}:  P(scatE=1)")
        fig.colorbar(im, ax=axs[row, col])

    resid = pred - obs
    good = np.isfinite(resid)
    rms = np.sqrt(np.average(resid[good] ** 2, weights=n[good]))
    im = draw_map(axs[row, 2], xe, ye, resid, *pair, "RdBu_r", vmin=-lim, vmax=lim)
    axs[row, 2].set_title(f"MLP - measured   (weighted RMS {rms:.4f})")
    fig.colorbar(im, ax=axs[row, 2])

fig.suptitle(
    f"2-D efficiency maps and closure residuals, {len(y_true)} test events   "
    f"(grey = fewer than {MIN_COUNT} events/bin; "
    f"red = over-predicts, blue = under-predicts, saturating at +/-{lim:.3f})"
)
fig.tight_layout()
plt.show()

for pair in pairs:
    xe, ye, n, obs, pred = maps[pair]
    fx, fy = (t.INPUT_FIELDS[i] for i in pair)
    resid = pred - obs
    g = np.isfinite(resid)
    print(
        f"({fx}, {fy}): {int((n >= MIN_COUNT).sum())} bins kept, "
        f"{n[n >= MIN_COUNT].sum() / n.sum() * 100:.2f}% of events   "
        f"weighted RMS {np.sqrt(np.average(resid[g] ** 2, weights=n[g])):.4f}   "
        f"max |residual| {np.nanmax(np.abs(resid)):.4f}"
    )